In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy qiskit-addon-cutting
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

*Estimare de utilizare: un minut pe un procesor Eagle (NOTĂ: Aceasta este doar o estimare. Timpul tău de execuție poate varia.)*

## Context

Circuit-knitting este un termen general care înglobează diverse metode de partiționare a unui circuit în mai multe subcircuite mai mici, implicând mai puține porți și/sau qubiți. Fiecare dintre subcircuite poate fi executat independent, iar rezultatul final se obține prin procesare clasică ulterioară a rezultatelor fiecărui subcircuit. Această tehnică este accesibilă prin [addon-ul Qiskit pentru tăierea circuitelor](https://qiskit.github.io/qiskit-addon-cutting/index.html); o explicație detaliată a tehnicii se găsește în [documentație](https://qiskit.github.io/qiskit-addon-cutting/explanation/index.html), alături de alt [material introductiv](https://qiskit.github.io/qiskit-addon-cutting/tutorials/index.html).

Acest notebook se ocupă de o metodă numită <b>tăierea firelor</b>, în care circuitul este partiționat de-a lungul firului [\[1\], \[2\]](#references). De remarcat că partiționarea este simplă în circuitele clasice, deoarece rezultatul la punctul de partiționare poate fi determinat determinist și este fie 0, fie 1. Cu toate acestea, starea qubitului la punctul de tăiere este, în general, o stare mixtă. Prin urmare, fiecare subcircuit trebuie măsurat de mai multe ori în baze diferite (de obicei un set tomografic complet de baze, cum ar fi baza Pauli [\[3\], \[4\]](#references)), și pregătit corespunzător în propria sa stare proprie. Figura de mai jos (<i>cu amabilitatea: Teză de doctorat, Ritajit Majumdar</i>) arată un exemplu de tăiere a firelor pentru o stare GHZ cu 4 qubiți în trei subcircuite. Aici $M_j$ denotă un set de baze (de obicei Pauli X, Y și Z), iar $P_i$ denotă un set de stări proprii (de obicei $|0\rangle$, $|1\rangle$, $|+\rangle$ și $|+i\rangle$).

![wc-1.png](../docs/images/tutorials/wire-cutting-to-improve-performance/0ce8857b-7f5f-400e-8536-6a496c724d50.avif)
![wc-2.png](../docs/images/tutorials/wire-cutting-to-improve-performance/cbce4455-4794-4c81-8630-3e3993e1b29f.avif)

Deoarece fiecare subcircuit are mai puțini qubiți și/sau porți, este de așteptat ca acestea să fie mai puțin susceptibile la zgomot. Acest notebook arată un exemplu în care această metodă poate fi utilizată pentru a suprima eficient zgomotul din sistem.

## Cerințe

Înainte de a începe acest tutorial, asigură-te că ai instalate următoarele:

- Qiskit SDK v2.0 sau mai recent, cu suport pentru [vizualizare](https://docs.quantum.ibm.com/api/qiskit/visualization)
- Qiskit Runtime v0.22 sau mai recent ( `pip install qiskit-ibm-runtime` )
- Addon-ul Qiskit pentru tăierea circuitelor v0.9.0 sau mai recent (`pip install qiskit-addon-cutting`)

Vom considera un circuit Many Body Localization (MBL) pentru acest notebook. Circuitul MBL este un circuit eficient pentru hardware și este parametrizat de doi parametri $\theta$ și $\vec{\phi}$. Când $\theta$ este setat la $0$ și starea inițială este pregătită în $|0\rangle$ pentru toți qubiții, valoarea de așteptare ideală a $\langle Z_i \rangle$ este $+1$ pentru fiecare site de qubit $i$, indiferent de valorile lui $\vec{\phi}$. Poți găsi mai multe detalii despre circuitele MBL în <a href="https://arxiv.org/abs/2307.07552">această lucrare</a>.

## Configurare

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from qiskit.circuit import Parameter, ParameterVector, QuantumCircuit
from qiskit.quantum_info import PauliList, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit.result import sampled_expectation_value

from qiskit_addon_cutting.instructions import CutWire
from qiskit_addon_cutting import (
    cut_wires,
    expand_observables,
    partition_problem,
    generate_cutting_experiments,
    reconstruct_expectation_values,
)

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime import SamplerV2, Batch

## Partea I. Exemplu la scară mică

### Pasul 1: Maparea intrărilor clasice la o problemă cuantică

Inițial construim un circuit șablon fără valori specifice ale parametrilor. Furnizăm, de asemenea, substituenți, numiți `CutWire`, pentru a adnota pozițiile tăieturilor. Pentru exemplul la scară mică considerăm un circuit MBL cu 10 qubiți.

In [2]:
class MBLChainCircuit(QuantumCircuit):
    def __init__(
        self, num_qubits: int, depth: int, use_cut: bool = False
    ) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainCircuit<{num_qubits}, {depth}>"
        )
        evolution = MBLChainEvolution(num_qubits, depth, use_cut)
        self.compose(evolution, inplace=True)


class MBLChainEvolution(QuantumCircuit):
    def __init__(self, num_qubits: int, depth: int, use_cut) -> None:
        super().__init__(
            num_qubits, name=f"MBLChainEvolution<{num_qubits}, {depth}>"
        )

        theta = Parameter("θ")
        phis = ParameterVector("φ", num_qubits)

        for layer in range(depth):
            layer_parity = layer % 2
            # print("layer parity", layer_parity)
            for qubit in range(layer_parity, num_qubits - 1, 2):
                # print(qubit)
                self.cz(qubit, qubit + 1)
                self.u(theta, 0, np.pi, qubit)
                self.u(theta, 0, np.pi, qubit + 1)
                if (
                    use_cut
                    and layer_parity == 0
                    and (
                        qubit == num_qubits // 2 - 1
                        or qubit == num_qubits // 2
                    )
                ):
                    self.append(CutWire(), [num_qubits // 2])
                if use_cut and layer < depth - 1 and layer_parity == 1:
                    if qubit == num_qubits // 2:
                        self.append(CutWire(), [qubit])
            for qubit in range(num_qubits):
                self.p(phis[qubit], qubit)

In [3]:
num_qubits = 10
depth = 2
mbl = MBLChainCircuit(num_qubits, depth)
mbl.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/a3debf65-06df-4277-933e-14b6f6170756-0.avif" alt="Output of the previous code cell" />

We calculate the average expectation value $O = \frac{1}{n} \sum_i Z_i$ over all qubits for $\theta = 0$. Since the ideal expectation value of $\langle Z_i \rangle = 1$ $\forall$ $i$, the ideal expectation value of $O$ is also $1$. The parameters $\phi$ are selected randomly.

In [4]:
np.random.seed(42)
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

Acum adnotăm circuitul pentru tăiere inserând **CutWire**-uri corespunzătoare pentru a crea două tăieturi aproximativ egale. Setăm `use_cut=True` în funcție și permitem adnotarea după $\frac{n}{2}$ qubiți, unde $n$ este numărul de qubiți din circuitul original.

In [5]:
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_cut.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/5208e0a8-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/31844134-514b-46ea-85f9-133e432f053f-0.avif)

### Pasul 2: Optimizarea problemei pentru execuția pe hardware cuantic
În continuare tăiem circuitul în două subcircuite mai mici. Pentru acest exemplu, ne limităm la doar 2 subcircuite. Pentru aceasta, folosim <a href="https://qiskit.github.io/qiskit-addon-cutting/">Addon-ul Qiskit: Tăierea Circuitelor</a>.
#### Tăierea circuitului în subcircuite mai mici
Tăierea firului la un punct crește numărul de qubiți cu unu. Pe lângă qubitul original, există acum un qubit suplimentar ca substituent în circuitul de după tăiere. Imaginea următoare oferă o reprezentare:

![wc-4.png](../docs/images/tutorials/wire-cutting-to-improve-performance/dfc5f923-e507-4873-888e-d90e1618be3a.avif)

Acest Addon folosește funcția `cut_wires` pentru a ține cont de qubiții suplimentari apăruți în urma tăierii.

In [6]:
mbl_move = cut_wires(mbl_cut)
mbl_move.draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1834cb22-0.avif" alt="Output of the previous code cell" />

#### Crearea și extinderea observabililor
Acum construim observabilul $M_z = \frac{1}{n}\sum_{i=1}^n \langle Z_i \rangle$. Deoarece rezultatul ideal al $\langle Z_i \rangle$ pentru fiecare $i$ este $+1$, rezultatul ideal al lui $M_z$ este, de asemenea, $+1$.

In [7]:
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
observable

PauliList(['ZIIIIIIIII', 'IZIIIIIIII', 'IIZIIIIIII', 'IIIZIIIIII',
           'IIIIZIIIII', 'IIIIIZIIII', 'IIIIIIZIII', 'IIIIIIIZII',
           'IIIIIIIIZI', 'IIIIIIIIIZ'])

In [8]:
new_obs = expand_observables(observable, mbl, mbl_move)
new_obs

PauliList(['ZIIIIIIIIII', 'IZIIIIIIIII', 'IIZIIIIIIII', 'IIIZIIIIIII',
           'IIIIZIIIIII', 'IIIIIIZIIII', 'IIIIIIIZIII', 'IIIIIIIIZII',
           'IIIIIIIIIZI', 'IIIIIIIIIIZ'])

Cu toate acestea, de remarcat că numărul de qubiți din circuit a crescut după inserarea operațiilor virtuale `Move` cu 2 qubiți în urma tăierii. Prin urmare, trebuie să extindem și observabilii prin inserarea de identități pentru a corespunde circuitului curent.

In [12]:
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

Here we visualize the two subcircuits:

In [13]:
subcircuits[0].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/1b0e779f-0.avif" alt="Output of the previous code cell" />

In [14]:
subcircuits[1].draw("mpl", fold=-1)

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/3c802f28-0.avif" alt="Output of the previous code cell" />

Să vizualizăm subcircuitele

In [15]:
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

As discussed before, for each cut the upstream circuit must be measured in a Pauli basis, and the downstream circuit must be prepared in the eigenstate of the basis. The function `generate_cutting_experiments` creates all of these necessary circuits and the coefficients associated with each circuit required for reconstruction. Find more detail in [this paper](https://journals.aps.org/prl/abstract/10.1103/PhysRevLett.125.150504).

In [16]:
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

#### Transpile the circuits onto the backend

For the first example involving only simulation, we transpile the circuit into the basis gate set of the backend:

In [17]:
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=133
)

print(backend)

<IBMBackend('ibm_fez')>


![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/35920640-76e8-4af6-a252-ee6a22e9c26a-0.avif)

Observabilii au fost și ei partiționați pentru a se potrivi subcircuitelor

In [20]:
pm_basis = generate_preset_pass_manager(
    optimization_level=2, basis_gates=backend.configuration().basis_gates
)
basis_subexperiments = {
    label: pm_basis.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

In [21]:
sampler = SamplerV2(mode=AerSimulator())
jobs = {
    label: sampler.run(subsystem_subexpts, shots=2**12)
    for label, subsystem_subexpts in basis_subexperiments.items()
}

De remarcat că fiecare subcircuit generează un număr de eșantioane. Reconstrucția ia în considerare rezultatul fiecăruia dintre aceste eșantioane. Fiecare dintre aceste eșantioane este denumit un `subexperiment`.
Extinderea observabilului folosind operația `Move` necesită o structură de date `PauliList`. Putem crea de asemenea observabilul $M_z$ în structura de date mai generică `SparsePauliOp`, care va fi utilă ulterior la reconstrucția subexperimentelor.

In [22]:
# Retrieve results
results = {label: job.result() for label, job in jobs.items()}

In [23]:
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real
reconstructed_expval

np.float64(0.9953821063041687)

In [32]:
methods = [
    "Uncut",
    "Wire cut",
]
values = [
    1,
    reconstructed_expval,
]  # since the ideal expectation value in noiseless simulation is +1

ax = plt.gca()
plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
ax.set_ylabel(r"$M_Z$", fontsize=12)

Text(0, 0.5, '$M_Z$')

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/fb5f955a-1.avif" alt="Output of the previous code cell" />

Să vedem două exemple în care qubiții tăiați sunt măsurați în două baze diferite. Mai întâi, este măsurat în baza normală Z, iar apoi în baza X.

In [ ]:
num_qubits = 60
depth = 2

# construct the circuit
mbl = MBLChainCircuit(num_qubits, depth)

# create parameters
phis = list(np.random.rand(mbl.num_parameters - 1))
theta = [0]
params = theta + phis

# construct the cut circuit
mbl_cut = MBLChainCircuit(num_qubits, depth, use_cut=True)
mbl_cut.assign_parameters(params, inplace=True)
mbl_move = cut_wires(mbl_cut)

# Define observable and expand to account for the wire cut
observable = PauliList(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)]
)
new_obs = expand_observables(observable, mbl, mbl_move)

# Construct a SparsePauliOp version of the observable for later use in reconstruction
M_z = SparsePauliOp(
    ["I" * i + "Z" + "I" * (num_qubits - i - 1) for i in range(num_qubits)],
    coeffs=[1 / num_qubits] * num_qubits,
)

# Partition the circuit and get subcircuits and subobservables
partitioned_problem = partition_problem(circuit=mbl_move, observables=new_obs)
subcircuits = partitioned_problem.subcircuits
subobservables = partitioned_problem.subobservables

# Obtain subexperiments and coefficients
subexperiments, coefficients = generate_cutting_experiments(
    circuits=subcircuits,
    observables=subobservables,
    num_samples=np.inf,
)

# Transpile the subexperiments to the backend
pm = generate_preset_pass_manager(optimization_level=2, backend=backend)
isa_subexperiments = {
    label: pm.run(partition_subexpts)
    for label, partition_subexpts in subexperiments.items()
}

# Execute the subexperiments and retrieve results
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    sampler.options.environment.job_tags = ["TUT_WC"]
    jobs = {
        label: sampler.run(subsystem_subexpts, shots=2**12)
        for label, subsystem_subexpts in isa_subexperiments.items()
    }
results = {label: job.result() for label, job in jobs.items()}

# Reconstruct the expectation value of the original observable
reconstructed_expval_terms = reconstruct_expectation_values(
    results,
    coefficients,
    subobservables,
)
reconstructed_expval = np.dot(reconstructed_expval_terms, M_z.coeffs).real

# Compute the uncut circuit to obtain the noisy expectation value for comparison
sampler = SamplerV2(mode=backend)
sampler.options.environment.job_tags = ["TUT_WC"]

if mbl.num_clbits == 0:
    mbl.measure_all()
isa_mbl = pm.run(mbl)

pub = (isa_mbl, params)
uncut_job = sampler.run([pub])

uncut_counts = uncut_job.result()[0].data.meas.get_counts()
uncut_expval = sampled_expectation_value(uncut_counts, M_z)

# visualize the results
ax = plt.gca()
methods = ["uncut", "cut"]
values = [uncut_expval, reconstructed_expval]

plt.bar(methods, values, color="#a56eff", width=0.4, edgecolor="#8a3ffc")
plt.axhline(y=1, color="k", linestyle="--")
plt.text(0.3, 0.95, "Exact result")
plt.show()

<Image src="../docs/images/tutorials/wire-cutting/extracted-outputs/37834c72-0.avif" alt="Output of the previous code cell" />

In [29]:
uncut_expval

0.9202473958333336

![Output of the previous code cell](../docs/images/tutorials/wire-cutting/extracted-outputs/987547e4-296a-41e4-ad82-41f4139a87a0-0.avif)

#### Transpilarea fiecărui subexperiment

În prezent trebuie să transpilăm circuitele noastre înainte de a le trimite spre execuție. Prin urmare, vom transpila mai întâi fiecare circuit din subexperimente.